## Ingest Races Data (CSV to Delta Lake)

This notebook reads the **races.csv** file (containing Formula 1 race details) from the landing volume and loads it into a Bronze Delta table for downstream processing.

**What we do here:**
1. **Define schema** and **read** the CSV file
2. **Enrich** with `ingestion_timestamp` and `source_file` columns
3. **Write** to the Bronze Delta table `formula1.bronze.races`

#### Load Environment Configuration
Before we begin, we run two shared notebooks:
- **`01.environment-config`** - loads variable names (catalog, schemas, file paths)
- **`02.bronze_helpers`** - loads the helper function that adds metadata columns

This way we don't repeat the same setup code in every notebook.

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
source_file = f'{loding_folder_path}/races.csv' # to replace the load data in read api
table_name = f'{catalog_name}.{bronze_schema}.races' # to replace the save table in write api 

In [0]:
display(source_file)

#### Step 1: Define Schema and Read the CSV
We define an explicit schema for the races file - this tells Spark exactly what columns to expect and what data type each one holds.

**Our schema has 6 columns:**

| Column | Type | What it stores |
| --- | --- | --- |
| `season` | Integer | The year of the race (e.g., 2023) |
| `round` | Integer | Race number in that season (e.g., 1st race, 2nd race) |
| `url` | String | Wikipedia link for the race |
| `raceName` | String | Name of the Grand Prix (e.g., "British Grand Prix") |
| `date` | Date | When the race took place |
| `circuitId` | String | ID linking to which circuit the race was held at |

**Why define this manually?** Without it, Spark guesses types (and sometimes gets it wrong). For example, `circuitId` contains text values like "silverstone" - if Spark guessed Integer, it would fail.

In [0]:
from pyspark.sql.types import *
races_schema = StructType([
  StructField('season', IntegerType(), True),
  StructField('round', IntegerType(), True),
  StructField('url', StringType(), True),
  StructField('raceName',StringType(), True),
   StructField('date', DateType(), True),
  StructField('circuitId', StringType(), True)
 ])

In [0]:
races_df = (
  spark.read
  .format('csv')
  .option('header', True)
  .schema(races_schema)
  .load(source_file))
display(races_df)

#### Step 2: Add Metadata Columns
We add two audit columns using the shared helper function `add_ingerstion_metadata()`:
- **`ingestion_timestamp`** - captures *when* this data was loaded into the lakehouse
- **`source_file`** - captures *which file* each row originated from

**Why?** If something looks wrong later, these columns help you trace back to when and where the data came from.

In [0]:
races_final_df = add_ingestion_metadata(races_df)
display(races_final_df)

#### Step 3: Write to Bronze Delta Table
We save the enriched DataFrame as a Delta table (`formula1.bronze.races`).

**What's happening:**
- `format('delta')` - saves in Delta format (versioning, time travel, fast queries)
- `mode('overwrite')` - replaces the table each run (clean reload from source)
- `saveAsTable(...)` - registers it in Unity Catalog so anyone can query it

In [0]:
(
  races_final_df
  .write
  .format('delta')
  .mode('overwrite')
  .option('overwriteSchema', 'true')
  .saveAsTable(table_name)
)